# Data Leakage Analysis (Startup Case)

This notebook demonstrates how to check for target and entity leakage between train and test splits, simulating a real-world startup scenario. The approach is based on guardrails for tabular ML pipelines.

In [ ]:
# Install dependencies if running in Colab
!pip install pandas numpy scikit-learn

## Leakage Detection Function

This function computes the intersection of entities, duplicate rows, and target drift between a training dataset and a test test.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

def valuta_data_leakage(
    df_train: pd.DataFrame, df_test: pd.DataFrame, colonna_id=None, colonna_target=None
):
    """
    Esegue i controlli standard per il data leakage tra training e test set.
    """
    report = {}

    # 1. Leakage di Riga (Duplicati esatti)
    # Rimuoviamo il target per assicurarci di trovare le stesse feature
    # (anche se per errore hanno label diverse, è comunque un problema)
    cols_to_check = [c for c in df_train.columns if c != colonna_target]

    # Unione interna per trovare le righe identiche in base alle feature
    intersezione = pd.merge(
        df_train[cols_to_check], df_test[cols_to_check], how="inner"
    )

    # Rimuoviamo i duplicati interni per avere il conteggio reale delle righe leakate
    righe_leaked = len(intersezione.drop_duplicates())
    report["Righe esatte duplicate (Leakage)"] = righe_leaked
    report["% Leakage su Test Set"] = (righe_leaked / len(df_test)) * 100 if len(df_test) > 0 else 0

    # 2. Entity Leakage (Sovrapposizione di ID)
    if colonna_id and colonna_id in df_train.columns:
        id_train = set(df_train[colonna_id].dropna().unique())
        id_test = set(df_test[colonna_id].dropna().unique())

        entita_condivise = id_train.intersection(id_test)
        report["Entità (ID) condivise"] = len(entita_condivise)

    # 3. Target Drift (Controllo statistico)
    if colonna_target and colonna_target in df_train.columns:
        media_train = df_train[colonna_target].mean()
        media_test = df_test[colonna_target].mean()
        report["Differenza media Target"] = abs(media_train - media_test)

    return report

## Simulate a Leaky Pipeline

Let's generate synthetic user data and intentionally introduce a leak by copying data across the train/test split boundary, then use our function to detect it.

In [ ]:
# 1. Creiamo un dataset fittizio
X, y = make_classification(n_samples=5000, n_features=20, random_state=42)
df = pd.DataFrame(X, columns=[f"feat_{i}" for i in range(20)])
df["target"] = y
df["user_id"] = np.random.randint(1, 1000, size=5000)  # Simuliamo degli ID

# 2. Split (introduciamo volutamente un po' di leakage per testare la funzione)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# INSERIAMO DEL LEAKAGE VOLONTARIO: copiamo 50 righe dal train al test
test_df = pd.concat([test_df, train_df.head(50)])

# 3. Valutazione Leakage
print("--- Analisi Leakage Pre-Training ---")
risultati = valuta_data_leakage(
    train_df, test_df, colonna_id="user_id", colonna_target="target"
)

for metrica, valore in risultati.items():
    if isinstance(valore, float):
        print(f"{metrica}: {valore:.2f}")
    else:
        print(f"{metrica}: {valore}")

# 4. Conclusioni
if risultati["Righe esatte duplicate (Leakage)"] == 0:
    print("\nNessun leakage rilevato. Avvio addestramento...")
else:
    print("\nATTENZIONE: Rilevato data leakage! Pulire il dataset prima di procedere.")